# SLT + RL — Kaggle Training Notebook (chạy 1 lệnh/subset cho TOÀN BỘ ma trận)

**Setup**: Settings → Accelerator: **GPU T4 x2** (hoặc P100). Internet: ON.
**Datasets cần add** (Add Input):
1. `phoenix-2014t` — annotation csv + (không cần fullFrame ảnh ở đây, chỉ cần csv).
2. `phoenix-poses` — pose đã extract sẵn từ `KAGGLE_NOTEBOOK_EXTRACT.ipynb` (notebook RIÊNG, chạy
   trên kernel CPU-only trước — xem file đó, không extract pose ở notebook train này).
3. `slt-rl-code` — thư mục code này, upload thành Kaggle Dataset riêng (hoặc clone từ GitHub ở Cell 2).

**Cách dùng notebook này**: chỉ có 3 lệnh chạy thật sự (Cell 4/5/6), mỗi lệnh chạy **TOÀN BỘ**
ma trận thí nghiệm (baseline sàn, 6 encoder, 5 thuật toán RL, reward ablation,
REINFORCE/A2C/Curriculum ablation, P7 two-stage, selection/decode policy, latency) cho **một**
subset. Không cần chạy cell-by-cell cho từng thí nghiệm con nữa — `run_all.py` tự làm hết.

Bạn KHÔNG bắt buộc chạy cả 3 dòng trong 1 lần ngồi: chạy Cell 4 (25%) trước, xem kết quả, rồi
quyết định khi nào chạy Cell 5 (50%)/Cell 6 (100%) — có thể là phiên Kaggle khác, tuần khác.
Vì `xe_epochs`/`rl_epochs` **không giảm theo subset** (đã chốt), subset 100% có thể mất nhiều hơn
1 session 12h — cứ bấm CHẠY LẠI ĐÚNG CELL ĐÓ, `run_all.py` tự resume (bỏ qua phần đã xong nhờ
marker file `.done_*` trong mỗi thư mục log) chứ không chạy lại từ đầu.

Chi tiết từng nhóm thí nghiệm ↔ mục lý thuyết: xem `docs/1_Thuyet_Trinh_Tong_Hop.md §K` và `run_all.py` (docstring đầu file).

## Cell 1 — Cài đặt dependencies
`bert-score` chỉ cần nếu bật `reward_bert_weight>0` (Experiment 9) — cài luôn cho tiện, nhẹ.

In [ ]:
!pip install -q sentencepiece sacrebleu einops bert-score


## Cell 2 — Lấy code
Cách 1: clone từ GitHub. Cách 2: copy từ Kaggle Dataset đã upload thư mục `slt_rl/`.

In [ ]:
import os
os.chdir("/kaggle/working")
# Cách 1: nếu push code lên github
# !git clone https://github.com/<ban>/slt_rl.git
# Cách 2: upload thư mục slt_rl thành Kaggle Dataset rồi copy
!cp -r /kaggle/input/slt-rl-code/slt_rl /kaggle/working/
os.chdir("/kaggle/working/slt_rl")


## Cell 3 — Kiểm tra pose cache đã mount đúng chưa
Phải ra số > 0 (số file `.npz`, khớp số câu đã extract ở `KAGGLE_NOTEBOOK_EXTRACT.ipynb`) — nếu ra 0 thì dừng lại và kiểm tra lại Add Input trước khi chạy bất cứ gì bên dưới (dataset PHOENIX-2014T không có video, model sẽ học từ toàn vector 0 mà không crash nếu path sai).

In [ ]:
import os
pose_dir = "/kaggle/input/phoenix-poses"
print(f"Pose files: {len(os.listdir(pose_dir))}")


## Cell 4 — Chạy TOÀN BỘ ma trận thí nghiệm, subset 25%

Chạy TRƯỚC TIÊN. Bao gồm: baseline sàn, core (Transformer+SCST XE→RL), 5 encoder còn lại
(P2-P6), 4 thuật toán RL còn lại (PPO/MRT/RAML/DPO), REINFORCE/A2C/Curriculum ablation, reward
ablation (4 tổ hợp), P7 two-stage, selection/decode policy, latency — TẤT CẢ cho subset 25%.

Nếu chỉ muốn test nhanh luồng trước khi chạy full, có thể giới hạn:
`!python run_all.py --subset 0.25 --groups core` rồi bỏ giới hạn ở lần chạy thật.

In [ ]:
!python run_all.py --subset 0.25


## Cell 5 — Chạy TOÀN BỘ ma trận, subset 50%

Chỉ chạy khi đã xem qua kết quả 25% và sẵn sàng dùng thêm quota GPU. Không phụ thuộc Cell 4 phải
chạy xong trước (subset là tập con độc lập của train), nhưng nên chạy sau để so sánh tăng dần
theo kích thước dữ liệu (Experiment 11).

In [ ]:
!python run_all.py --subset 0.5


## Cell 6 — Chạy TOÀN BỘ ma trận, subset 100%

Chạy SAU CÙNG. Epoch không giảm theo subset nên đây là lần chạy nặng nhất — nhiều khả năng cần
NHIỀU session Kaggle (12h/session). Hết giờ giữa chừng thì mở lại notebook, chạy LẠI ĐÚNG CELL
NÀY — `run_all.py` tự bỏ qua các bước đã xong (marker `.done_*`), không tốn công chạy lại.

In [ ]:
!python run_all.py --subset 1.0


## Cell 7 — Bảng so sánh cuối cùng giữa mọi phương pháp/subset

`run_all.py` đã tự gọi `scripts/aggregate_results.py` **và** `scripts/make_report.py` ở cuối mỗi
lần chạy, nên bảng/hình dưới đây LUÔN cập nhật tới thời điểm hiện tại kể cả khi mới chạy xong
Cell 4 (chưa chạy Cell 5/6). Chạy lại cell này bất cứ lúc nào để xem tình trạng mới nhất — không
cần gõ thêm lệnh gì khác.

- `comparison_table.csv/.md` — 1 bảng gộp MỌI run (thô, đầy đủ cột).
- `report/tables/*.csv/.md` — 6 bảng đã lọc sẵn theo từng câu hỏi so sánh (main/encoder/algo/
  reward ablation/ablation khác/baseline/latency) — dễ đọc hơn bảng gộp ở trên.
- `report/tables/tab_*.tex` — 3 bảng LaTeX dán thẳng vào `paper/sn-article.tex`
  (`tab:main`/`tab:reward`/`tab:encresults`). Cột BLEU-1/ROUGE-L để `--` vì pipeline hiện chỉ
  tính BLEU-4 (không bịa số).
- `report/figures/*.png/.pdf` — 5 biểu đồ so sánh (BLEU theo epoch, ΔBLEU theo subset, trade-off
  reward ablation, so sánh encoder, so sánh thuật toán). PNG xem nhanh, PDF vector để chèn paper.

In [ ]:
!python scripts/aggregate_results.py --work_dir /kaggle/working --out /kaggle/working/comparison_table
!python scripts/make_report.py --work_dir /kaggle/working
import pandas as pd
df = pd.read_csv("/kaggle/working/comparison_table.csv")
df.sort_values(["subset_pct", "encoder", "method"])


## Cell 7b — Xem nhanh 1 biểu đồ ngay trong notebook (tuỳ chọn)

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("/kaggle/working/report/figures/*.png")):
    print(f)
    display(Image(filename=f))


## Cell 8 — Nén output để tải về

In [ ]:
!tar czf /kaggle/working/all_logs.tar.gz /kaggle/working/run1_* /kaggle/working/rw_*_subset* \
    /kaggle/working/baseline_* /kaggle/working/p7_twostage_* /kaggle/working/report \
    /kaggle/working/comparison_table.csv /kaggle/working/comparison_table.md
